# Assignment 03 - Project: Deep Research with LangGraph - Research Agent
This notebook implements **Module 3: Research Agent - Research Agent with Custom Tools**. The agent uses a search tool to crawl details, summarizes findings, and decides whether to continue searching or finish.

In [1]:
import sys
import os
from dotenv import load_dotenv

sys.path.append(os.path.abspath(".."))
from mock_llm import get_llm

load_dotenv()
print("Environment loaded successfully.")

Environment loaded successfully.


In [2]:
from typing import List, TypedDict
from langgraph.graph import StateGraph, START, END

class ResearchState(TypedDict):
    brief: str
    current_query: str
    queries: List[str]
    search_results: str
    notes: str
    iteration: int

In [3]:
def mock_web_search(query: str) -> str:
    """Custom search tool to crawl facts."""
    query_lower = query.lower()
    if "nist" in query_lower:
        return "[NIST standard publication 2024] NIST finalized lattice-based standards ML-KEM and ML-DSA for post-quantum security."
    elif "shor" in query_lower or "quantum threat" in query_lower:
        return "[Academic Research] Shor's algorithm resolves RSA discrete logarithms in polynomial time, making RSA-2048 insecure once quantum power reaches 20 million physical qubits."
    else:
        return f"[Search Result] Standard industry reports for research query '{query}': Post-quantum migration planning is underway."


In [4]:
def research_planner(state: ResearchState):
    """Node: Formulate search query based on the brief and existing findings."""
    llm = get_llm(model="gpt-4o-mini", temperature=0)
    prompt = f"""Based on the research brief:
\"\"{state['brief']}\"\"
And existing research notes:
\"\"{state['notes']}\"\"
Formulate the next search query to run. Return ONLY the search query text."""
    
    res = llm.invoke(prompt)
    return {"current_query": res.content}

In [5]:
def search_executor(state: ResearchState):
    """Node: Execute search tool."""
    query = state["current_query"]
    results = mock_web_search(query)
    return {
        "search_results": results,
        "queries": state["queries"] + [query]
    }

In [6]:
def notes_synthesizer(state: ResearchState):
    """Node: Append search results and refine research notes."""
    llm = get_llm(model="gpt-4o-mini", temperature=0)
    prompt = f"""Update and synthesize the research notes based on new search findings.
Current Notes:
{state['notes']}
New Findings:
{state['search_results']}
Write the updated, aggregated research report."""
    
    res = llm.invoke(prompt)
    return {
        "notes": res.content,
        "iteration": state["iteration"] + 1
    }

In [7]:
def should_continue(state: ResearchState):
    """Router: Limit iterations or end if brief goals are met."""
    if state["iteration"] >= 2:
        return END
    return "plan"

In [8]:
# Compile the graph
builder = StateGraph(ResearchState)
builder.add_node("plan", research_planner)
builder.add_node("search", search_executor)
builder.add_node("synthesize", notes_synthesizer)

builder.add_edge(START, "plan")
builder.add_edge("plan", "search")
builder.add_edge("search", "synthesize")
builder.add_conditional_edges("synthesize", should_continue, {"plan": "plan", END: END})

graph = builder.compile()

In [9]:
# Print the graph architecture
try:
    print(graph.get_graph().draw_ascii())
except Exception as e:
    print("Could not draw graph:", e)

        +-----------+     
        | __start__ |     
        +-----------+     
              *           
              *           
              *           
          +------+        
          | plan |        
          +------+.       
          *        ..     
        **           .    
       *              ..  
+--------+              . 
| search |            ..  
+--------+           .    
          *        ..     
           **    ..       
             *  .         
       +------------+     
       | synthesize |     
       +------------+     
              .           
              .           
              .           
         +---------+      
         | __end__ |      
         +---------+      


In [10]:
brief_text = "# Brief: Shor's algorithm threat and NIST lattice standards."
initial_state = {
    "brief": brief_text,
    "current_query": "",
    "queries": [],
    "search_results": "",
    "notes": "Initial notes: Cryptography secures current communications.",
    "iteration": 0
}

print("--- Executing Research Agent Loop ---")
res = graph.invoke(initial_state)

print("\nExecuted Search Queries:", res["queries"])
print("\n--- Final Research Notes ---")
print(res["notes"])

--- Executing Research Agent Loop ---


--- Using Live Ollama Cloud (gpt-oss:120b) ---


--- Using Live Ollama Cloud (gpt-oss:120b) ---


--- Using Live Ollama Cloud (gpt-oss:120b) ---


--- Using Live Ollama Cloud (gpt-oss:120b) ---



Executed Search Queries: ["Shor's algorithm threat to RSA and ECC and analysis of NIST post‑quantum lattice‑based standardization progress.", 'estimated quantum resources and timeline required to break RSA‑2048 and ECC‑P256 using Shor’s algorithm (2024‑2026 studies)']

--- Final Research Notes ---
**Research Report – Post‑Quantum Cryptography Landscape (2024‑2025 – Updated)**  
*Prepared: 12 July 2026*  

---

## 1. Executive Summary  

The confidentiality, integrity, authentication, and non‑repudiation of today’s digital infrastructure depend on public‑key primitives whose security rests on mathematical problems that are *easy* for classical computers but *hard* for adversaries.  Shor’s algorithm demonstrates that a sufficiently large, fault‑tolerant quantum computer can solve integer‑factorisation and discrete‑logarithm problems in polynomial time, instantly breaking RSA, Diffie‑Hellman, and elliptic‑curve schemes.  

Recent quantum‑hardware road‑maps (IBM, Google, and the Chinese “